# CAC 40 — IVol Spike Exit: Backtest Engine

Numba-accelerated IVol spike-exit / EMA re-entry strategy with full parameter grid scan.

**Signal logic:**
- **Exit**: go flat when IVol > EMA-N × (1 + spike%) for `exit_confirm` consecutive days (with `cooldown` anti-whipsaw)
- **Re-entry**: re-enter when EMA-fast < EMA-slow for `confirm` consecutive days

**Depends on**: `explore_ivol_signal.ipynb` having been run first for data (`clean`, `bh_eq`, `bh_sharpe`, `bh_total`, `bh_mdd`),  
or run the data cell below standalone.

In [1]:
import sys
import pandas as pd, numpy as np
import numba as nb
from itertools import product
import time
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

# ── Signum charting ──
sys.path.insert(0, r"C:\Personal\Business & Investments\Python codes")
from signum import Chart, Dashboard

import sfera_db

PROJECT_ROOT = Path(r"c:\Personal\Business & Investments\Trading portfolio\Cogilator\btest")
print("Ready")

Ready


In [2]:
# ── Load data (standalone — skip if already loaded from explore_ivol_signal) ──
cac_px   = sfera_db.index_prices('CAC')[['close', 'volume']]
cac_ivol = sfera_db.index_ivol('CAC')[['ivol']]

common_idx = cac_px.index.intersection(cac_ivol.index)
cac = pd.DataFrame({
    'close': cac_px.loc[common_idx, 'close'],
    'ivol':  cac_ivol.loc[common_idx, 'ivol'],
}).sort_index()
cac['ret'] = cac['close'].pct_change()
clean = cac.dropna(subset=['ret']).copy()
clean['ivol_ema5']  = clean['ivol'].ewm(span=5).mean()
clean['ivol_ema20'] = clean['ivol'].ewm(span=20).mean()

bh_eq     = (1 + clean['ret']).cumprod()
bh_total  = (bh_eq.iloc[-1] - 1) * 100
bh_sharpe = clean['ret'].mean() / clean['ret'].std() * np.sqrt(252)
bh_mdd    = ((bh_eq - bh_eq.cummax()) / bh_eq.cummax()).min() * 100

print(f"Data: {clean.index[0].date()} \u2192 {clean.index[-1].date()}  ({len(clean)} rows)")
print(f"B&H  Sharpe {bh_sharpe:.3f}  Total {bh_total:+.1f}%  MaxDD {bh_mdd:.1f}%")

Data: 2007-01-03 → 2026-03-20  (4911 rows)
B&H  Sharpe 0.182  Total +36.5%  MaxDD -59.2%


In [3]:
# ── Numba backtest kernels ──

@nb.njit(cache=True)
def _bt_core(ivol, ret, baseline, ema_f, ema_s,
             spike_pct, exit_confirm, cooldown, confirm):
    """Returns (sharpe, total%, maxdd%, pct_in%, n_transitions)."""
    spike_mult = 1.0 + spike_pct / 100.0
    n = len(ivol)
    pos = 1; prev_pos = 1; days_since_entry = cooldown
    days_ema_below = 0; days_above_spike = 0
    sum_ret = 0.0; sum_ret_sq = 0.0
    equity = 1.0; peak = 1.0; max_dd = 0.0
    days_in = 0; n_trans = 0

    for i in range(n):
        iv = ivol[i]
        thr_i = baseline[i] * spike_mult
        if iv >= thr_i: days_above_spike += 1
        else: days_above_spike = 0
        if ema_f[i] < ema_s[i]: days_ema_below += 1
        else: days_ema_below = 0

        old_pos = pos
        if pos == 1 and days_above_spike >= exit_confirm and days_since_entry >= cooldown:
            pos = 0
        elif pos == 0 and days_ema_below >= confirm:
            pos = 1; days_since_entry = 0
        if pos == 1: days_since_entry += 1
        if pos != old_pos: n_trans += 1

        strat_r = prev_pos * ret[i]
        sum_ret += strat_r; sum_ret_sq += strat_r * strat_r
        equity *= (1.0 + strat_r)
        if equity > peak: peak = equity
        dd = (equity - peak) / peak
        if dd < max_dd: max_dd = dd
        if prev_pos > 0: days_in += 1
        prev_pos = pos

    total  = (equity - 1.0) * 100.0
    mean_r = sum_ret / n
    var_r  = sum_ret_sq / n - mean_r * mean_r
    std_r  = var_r ** 0.5 if var_r > 0.0 else 1e-10
    sharpe = mean_r / std_r * (252.0 ** 0.5)
    pct_in = days_in / n * 100.0
    return sharpe, total, max_dd * 100.0, pct_in, float(n_trans)


@nb.njit(cache=True)
def _bt_full(ivol, ret, baseline, ema_f, ema_s,
             spike_pct, exit_confirm, cooldown, confirm):
    """Returns full arrays for charting: (positions, exit_sig, entry_sig, exit_level)."""
    spike_mult = 1.0 + spike_pct / 100.0
    n = len(ivol)
    positions  = np.empty(n, dtype=np.float64)
    exit_sig   = np.empty(n, dtype=np.float64)
    entry_sig  = np.empty(n, dtype=np.float64)
    exit_level = np.empty(n, dtype=np.float64)
    pos = 1; days_since_entry = cooldown
    days_ema_below = 0; days_above_spike = 0

    for i in range(n):
        iv    = ivol[i]
        thr_i = baseline[i] * spike_mult
        if iv >= thr_i: days_above_spike += 1
        else: days_above_spike = 0
        if ema_f[i] < ema_s[i]: days_ema_below += 1
        else: days_ema_below = 0

        exit_sig[i]   = 0.0 if days_above_spike >= exit_confirm else 1.0
        entry_sig[i]  = 1.0 if days_ema_below   >= confirm      else 0.0
        exit_level[i] = thr_i

        if pos == 1 and days_above_spike >= exit_confirm and days_since_entry >= cooldown:
            pos = 0
        elif pos == 0 and days_ema_below >= confirm:
            pos = 1; days_since_entry = 0
        if pos == 1: days_since_entry += 1
        positions[i] = float(pos)

    return positions, exit_sig, entry_sig, exit_level


# ── Pre-compute EMA cache ──
ivol_arr = clean['ivol'].values.astype(np.float64)
ret_arr  = clean['ret'].values.astype(np.float64)

all_spans = sorted(set(range(2, 16)) | set(range(15, 45, 5)) | set(range(20, 125, 5)))
ema_cache = {s: clean['ivol'].ewm(span=s).mean().values.astype(np.float64) for s in all_spans}
print(f"Pre-computed {len(ema_cache)} EMA spans: {min(all_spans)}-{max(all_spans)}")

# Warm-up JIT
_ = _bt_core(ivol_arr, ret_arr, ema_cache[60], ema_cache[5], ema_cache[20], 30.0, 3, 15, 5)
_ = _bt_full(ivol_arr, ret_arr, ema_cache[60], ema_cache[5], ema_cache[20], 30.0, 3, 15, 5)
print("Numba JIT compiled \u2713")


def bt_ivol_spike(df, exit_ema=60, spike_pct=30, exit_confirm=3,
                  ema_fast=5, ema_slow=20, cooldown=15, confirm=5):
    """Full backtest: returns (result_df, stats_dict)."""
    res      = df[['close', 'ret', 'ivol']].copy()
    baseline = res['ivol'].ewm(span=exit_ema).mean().values.astype(np.float64)
    ema_f_a  = res['ivol'].ewm(span=ema_fast).mean().values.astype(np.float64)
    ema_s_a  = res['ivol'].ewm(span=ema_slow).mean().values.astype(np.float64)

    positions, exit_sig, entry_sig, exit_lev = _bt_full(
        res['ivol'].values.astype(np.float64), res['ret'].values.astype(np.float64),
        baseline, ema_f_a, ema_s_a, float(spike_pct), exit_confirm, cooldown, confirm)

    res['position']    = positions
    res['exit_sig']    = exit_sig
    res['entry_sig']   = entry_sig
    res['exit_level']  = exit_lev
    res['strat_ret']   = res['position'].shift(1).fillna(1) * res['ret']
    res['equity_strat'] = (1 + res['strat_ret']).cumprod()

    sr     = res['strat_ret']; eq = res['equity_strat']
    sharpe = sr.mean() / sr.std() * np.sqrt(252) if sr.std() > 0 else 0
    mdd    = ((eq - eq.cummax()) / eq.cummax()).min()
    total  = (eq.iloc[-1] - 1) * 100
    pct_in = (res['position'].shift(1).fillna(1) > 0).sum() / len(res) * 100
    stats  = dict(sharpe=round(sharpe, 3), total=round(total, 1),
                  maxdd=round(mdd * 100, 1), pct_in=round(pct_in, 1))
    return res, stats

Pre-computed 35 EMA spans: 2-120
Numba JIT compiled ✓


In [4]:
# ── Default run + Signum dashboard ──
res, stats = bt_ivol_spike(clean, exit_ema=60, spike_pct=30, exit_confirm=3,
                            ema_fast=5, ema_slow=20, cooldown=15, confirm=5)
print(f"Strategy: {stats}")
print(f"B&H:      Sharpe={bh_sharpe:.3f}  Total={bh_total:+.1f}%  MaxDD={bh_mdd:.1f}%")

pos_shifted  = res['position'].shift(1).fillna(1)
signal_marks = pd.DataFrame({'time': res.index,
                              'signal': res['position'].diff().fillna(0).astype(int)})
pos_shade    = pd.DataFrame({'time': res.index, 'position': pos_shifted.values})

t = res.index

p1 = (
    Chart(height=300)
    .line(pd.DataFrame({'time': t, 'value': res['close'].values}),
          name='CAC 40', color='#60a5fa', width=2)
    .signals(signal_marks, signal_col='signal',
             buy_text='IN', sell_text='OUT',
             buy_color='#34d399', sell_color='#f87171')
    .shade(pos_shade, position_col='position', color='#34d399', opacity=0.06)
)

p2 = (
    Chart(height=200)
    .line(pd.DataFrame({'time': t, 'value': res['equity_strat'].values}),
          name='Strategy', color='#34d399', width=2)
    .line(pd.DataFrame({'time': bh_eq.index, 'value': bh_eq.values}),
          name='Buy & Hold', color='#6b7280', width=1)
)

p3 = (
    Chart(height=200)
    .line(pd.DataFrame({'time': t, 'value': res['ivol'].values}),
          name='IVol', color='#fb923c', width=1)
    .line(pd.DataFrame({'time': t, 'value': clean['ivol_ema5'].values}),
          name='EMA-5', color='#fbbf24', width=1)
    .line(pd.DataFrame({'time': t, 'value': res['exit_level'].values}),
          name='Exit Level', color='#38bdf8', width=1)
)

p4 = Chart(height=100).baseline(
    pd.DataFrame({'time': t, 'value': res['exit_sig'].values}), base_value=0.5)

Dashboard(
    panes=[p1, p2, p3, p4],
    titles=[
        f'CAC 40  Sharpe {stats["sharpe"]:.3f}  Total {stats["total"]:+.1f}%  MaxDD {stats["maxdd"]:.1f}%',
        'Equity: Strategy vs B&H',
        'IVol + EMA-5 + Exit Level',
        'Exit Signal'
    ],
    theme='dark'
)

Strategy: {'sharpe': np.float64(0.228), 'total': np.float64(63.7), 'maxdd': np.float64(-52.8), 'pct_in': np.float64(96.6)}
B&H:      Sharpe=0.182  Total=+36.5%  MaxDD=-59.2%


In [5]:
# ── Full parameter grid scan (7 dimensions) ──
p_exit_ema     = [20, 30, 40, 50, 60, 80, 100, 120]
p_spike_pct    = list(range(15, 85, 5))
p_exit_confirm = list(range(1, 8))
p_cooldown     = [5, 10, 15, 20, 30]
p_ema_fast     = [2, 3, 4, 5, 6, 7, 8, 10]
p_ema_slow     = [15, 20, 25, 30]
p_confirm      = [1, 2, 3, 5, 7, 10, 15]

combos = list(product(p_exit_ema, p_spike_pct, p_exit_confirm,
                      p_cooldown, p_ema_fast, p_ema_slow, p_confirm))
print(f"Grid: {len(combos):,} combinations")

t0 = time.perf_counter()
results = np.empty((len(combos), 12), dtype=np.float64)

for idx, (ex_ema, sp, ex_cf, cd, ef, es, cf) in enumerate(combos):
    sh, tot, mdd, pin, ntr = _bt_core(
        ivol_arr, ret_arr, ema_cache[ex_ema], ema_cache[ef], ema_cache[es],
        float(sp), ex_cf, cd, cf)
    results[idx] = [ex_ema, sp, ex_cf, cd, ef, es, cf, sh, tot, mdd, pin, ntr]

elapsed = time.perf_counter() - t0
print(f"Done in {elapsed:.1f}s  ({elapsed/len(combos)*1e6:.1f} \u03bcs/backtest)")

opt = pd.DataFrame(results, columns=[
    'exit_ema', 'spike_pct', 'exit_confirm', 'cooldown',
    'ema_fast', 'ema_slow', 'confirm',
    'sharpe', 'total', 'maxdd', 'pct_in', 'n_trans'
])
for c in ['exit_ema','spike_pct','exit_confirm','cooldown','ema_fast','ema_slow','confirm','n_trans']:
    opt[c] = opt[c].astype(int)

viable = opt[(opt['pct_in'] > 50) & (opt['sharpe'] > 0)].copy()
viable['score'] = viable['sharpe'] - (viable['maxdd'].abs() / 100) * 0.5

show_cols = ['exit_ema','spike_pct','exit_confirm','cooldown',
             'ema_fast','ema_slow','confirm','sharpe','total','maxdd','pct_in','n_trans']
fmt = {'sharpe':'{:.3f}','total':'{:+.1f}%','maxdd':'{:.1f}%','pct_in':'{:.0f}%'}

print(f"\nViable (in-market > 50%, Sharpe > 0): {len(viable):,} / {len(opt):,}")
print("\nTop 20 by Sharpe:")
display(viable.nlargest(20, 'sharpe')[show_cols].reset_index(drop=True)
    .style.format(fmt)
    .bar(subset=['sharpe'], color='#66ffcc')
    .bar(subset=['total'], color='#8cb4ff')
    .background_gradient(subset=['maxdd'], cmap='RdYlGn'))

print("\nTop 10 Balanced (Sharpe \u2212 0.5\u00d7|MaxDD|):")
display(viable.nlargest(10, 'score')[show_cols + ['score']].reset_index(drop=True)
    .style.format({**fmt, 'score':'{:.3f}'}).bar(subset=['score'], color='#ffb347'))

Grid: 878,080 combinations
Done in 6.6s  (7.5 μs/backtest)

Viable (in-market > 50%, Sharpe > 0): 867,890 / 878,080

Top 20 by Sharpe:


,exit_ema,spike_pct,exit_confirm,cooldown,ema_fast,ema_slow,confirm,sharpe,total,maxdd,pct_in,n_trans
0,80,30,3,5,2,15,1,0.299,+113.7%,-51.0%,98%,20
1,60,30,3,5,5,20,7,0.286,+101.8%,-44.6%,96%,14
2,60,30,3,10,5,20,7,0.286,+101.8%,-44.6%,96%,14
3,60,30,3,15,5,20,7,0.286,+101.8%,-44.6%,96%,14
4,60,30,3,20,5,20,7,0.286,+101.8%,-44.6%,96%,14
5,60,30,3,30,5,20,7,0.286,+101.8%,-44.6%,96%,14
6,80,30,3,5,5,20,7,0.286,+101.8%,-44.6%,96%,14
7,80,30,3,10,5,20,7,0.286,+101.8%,-44.6%,96%,14
8,80,30,3,15,5,20,7,0.286,+101.8%,-44.6%,96%,14
9,80,30,3,20,5,20,7,0.286,+101.8%,-44.6%,96%,14



Top 10 Balanced (Sharpe − 0.5×|MaxDD|):


,exit_ema,spike_pct,exit_confirm,cooldown,ema_fast,ema_slow,confirm,sharpe,total,maxdd,pct_in,n_trans,score
0,60,30,3,5,5,20,7,0.286,+101.8%,-44.6%,96%,14,0.063
1,60,30,3,10,5,20,7,0.286,+101.8%,-44.6%,96%,14,0.063
2,60,30,3,15,5,20,7,0.286,+101.8%,-44.6%,96%,14,0.063
3,60,30,3,20,5,20,7,0.286,+101.8%,-44.6%,96%,14,0.063
4,60,30,3,30,5,20,7,0.286,+101.8%,-44.6%,96%,14,0.063
5,80,30,3,5,5,20,7,0.286,+101.8%,-44.6%,96%,14,0.063
6,80,30,3,10,5,20,7,0.286,+101.8%,-44.6%,96%,14,0.063
7,80,30,3,15,5,20,7,0.286,+101.8%,-44.6%,96%,14,0.063
8,80,30,3,20,5,20,7,0.286,+101.8%,-44.6%,96%,14,0.063
9,80,30,3,30,5,20,7,0.286,+101.8%,-44.6%,96%,14,0.063


In [6]:
# ── Visualise best result (Signum) ──
best = viable.nlargest(1, 'sharpe').iloc[0]
bp   = {k: int(best[k]) for k in ['exit_ema','spike_pct','exit_confirm',
                                    'cooldown','ema_fast','ema_slow','confirm']}
print(f"Best params: {bp}")

res_b, stats_b = bt_ivol_spike(clean, **bp)
pos_b   = res_b['position'].shift(1).fillna(1)
sig_b   = pd.DataFrame({'time': res_b.index,
                         'signal': res_b['position'].diff().fillna(0).astype(int)})
shade_b = pd.DataFrame({'time': res_b.index, 'position': pos_b.values})
t       = res_b.index

p1 = (
    Chart(height=300)
    .line(pd.DataFrame({'time': t, 'value': res_b['close'].values}),
          name='CAC 40', color='#60a5fa', width=2)
    .signals(sig_b, signal_col='signal', buy_text='IN', sell_text='OUT',
             buy_color='#34d399', sell_color='#f87171')
    .shade(shade_b, position_col='position', color='#34d399', opacity=0.06)
)
p2 = (
    Chart(height=200)
    .line(pd.DataFrame({'time': t, 'value': res_b['equity_strat'].values}),
          name='Strategy', color='#34d399', width=2)
    .line(pd.DataFrame({'time': bh_eq.index, 'value': bh_eq.values}),
          name='B&H', color='#6b7280', width=1)
)
p3 = (
    Chart(height=200)
    .line(pd.DataFrame({'time': t, 'value': res_b['ivol'].values}),
          name='IVol', color='#fb923c', width=1)
    .line(pd.DataFrame({'time': t, 'value': clean['ivol'].ewm(span=bp['ema_fast']).mean().values}),
          name=f'EMA-{bp["ema_fast"]}', color='#fbbf24', width=1)
    .line(pd.DataFrame({'time': t, 'value': res_b['exit_level'].values}),
          name=f'Exit (EMA-{bp["exit_ema"]}\u00d7{1+bp["spike_pct"]/100:.0%})', color='#38bdf8', width=1)
)

Dashboard(
    panes=[p1, p2, p3],
    titles=[
        f'BEST  Sharpe {stats_b["sharpe"]:.3f}  Total {stats_b["total"]:+.1f}%  MaxDD {stats_b["maxdd"]:.1f}%  In {stats_b["pct_in"]:.0f}%',
        'Equity: Strategy vs Buy & Hold',
        'IVol + Dynamic Exit Level'
    ],
    theme='dark'
)

Best params: {'exit_ema': 80, 'spike_pct': 30, 'exit_confirm': 3, 'cooldown': 5, 'ema_fast': 2, 'ema_slow': 15, 'confirm': 1}


---
## DSL Signal Equivalent

The IVol spike-exit logic maps naturally to the `quantdsl_backtest` DSL. Key nodes:

```python
from quantdsl_backtest.dsl.factors import VolatilityFactor
from quantdsl_backtest.dsl.signals import ZScoreRolling, Greater, Not, And, RiskMultiplierFromZ

factors = {
    # 20-day realised vol (annualised) — stands in for IVol
    "vol_20d": VolatilityFactor(name="vol_20d", field="close", lookback=20, annualize=True),
}
signals = {
    # Z-score of vol over 63-day rolling window
    "vol_z": ZScoreRolling(base="vol_20d", window=63, name="vol_z"),

    # Hard exit flag: vol z-score > 2.0
    "ivol_spike": Greater(left="vol_z", right=2.0, name="ivol_spike"),
    "no_spike":   Not(expr="ivol_spike", name="no_spike"),

    # Soft sizing: scale exposure down as vol rises (1.0 at z=0, 0.0 at z>=2.5)
    "vol_risk_mult": RiskMultiplierFromZ(z="vol_z", max_z=2.5, name="vol_risk_mult"),
}
```

**Note:** The DSL `VolatilityFactor` uses **realised vol from price** (Numba backtest uses **implied vol from options**).  
True IVol data is not yet a DSL data source — it would need a `TimeSeries` node pointing at the sfera_db IVol series.